# 03. Comparación de modelos con validación cruzada


Comparamos varios modelos con **KFold de 5 particiones** (barajado, semilla 42) y la métrica del enunciado, **RMSE**. Todos los modelos están definidos en `src/models.py`.

In [ ]:
import sys
from pathlib import Path

# Permite importar el paquete src/ desde la carpeta notebooks/
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from src.config import DATA_RAW, TARGET, CATEGORICAL, NUMERIC
sns.set_theme(style="whitegrid")
from src.preprocessing import add_features, split_xy
from src.models import get_models, comparar_modelos, evaluar_cv, rmse

In [ ]:
train = pd.read_csv(DATA_RAW / "train.csv")
X, y = split_xy(train)
X = add_features(X)
print("Baseline (predecir siempre la media):", round(rmse(y, np.full(len(y), y.mean())), 3))

## 1. Modelos candidatos

- **LinearRegression / Ridge:** lineales, rápidos e interpretables.
- **Ridge (log price):** Ridge sobre `log(price)` con corrección de smearing.
- **RandomForest:** muchos árboles promediados.
- **HistGradientBoosting:** boosting de scikit-learn.
- **LightGBM (log price):** boosting más ajustable, sobre `log(price)`.

Tarda unos minutos.

In [ ]:
resultados = comparar_modelos(get_models(), X, y)
resultados

In [ ]:
plt.figure(figsize=(8, 4))
plt.barh(resultados["modelo"], resultados["rmse_cv"], xerr=resultados["std"])
plt.gca().invert_yaxis()
plt.xlabel("RMSE CV (menor es mejor)"); plt.axvline(57, color="red", ls="--", label="umbral del 5 (57)")
plt.legend(); plt.show()

## 2. Análisis de errores del mejor modelo

In [ ]:
mejor = get_models()["Ridge (log price)"]
_, oof = evaluar_cv(mejor, X, y)
res = y - oof
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(oof, y, s=5, alpha=0.3); ax[0].plot([0, 1200], [0, 1200], "r--")
ax[0].set_xlabel("predicción"); ax[0].set_ylabel("real")
sns.histplot(res, bins=80, ax=ax[1]).set_title("residuos")
plt.show()
print("RMSE quitando los 1% de errores más grandes:", round(rmse(y[res.abs() < res.abs().quantile(0.99)], oof[res.abs() < res.abs().quantile(0.99)]), 2))

## 3. Conclusiones

- Los modelos sobre `log(price)` ganan a sus versiones en escala original.
- El mejor individual es **Ridge (log price)**, seguido de **LightGBM (log price)**. Que un modelo lineal gane a los árboles indica que la relación es casi lineal en escala logarítmica.
- Todos quedan por debajo del umbral de 57.
- Gran parte del error viene de unos pocos productos muy caros, difíciles de predecir.

En el notebook 04 se ajustan los dos mejores y se prueba mezclarlos.